In [0]:
%pip install locust

In [0]:
from databricks.sdk import WorkspaceClient

ENDPOINT_NAME = "databricks-claude-sonnet-4-6"

w = WorkspaceClient()
endpoint = w.serving_endpoints.get(ENDPOINT_NAME)

print(f"Endpoint: {endpoint.name}")
print(f"State: {endpoint.state}")
print()

if endpoint.ai_gateway and endpoint.ai_gateway.rate_limits:
    print("AI Gateway Rate Limits:")
    for rl in endpoint.ai_gateway.rate_limits:
        print(f"  - Key: {rl.key}, Renewal Period: {rl.renewal_period}, Calls: {rl.calls}, Tokens: {rl.tokens}, Principal: {rl.principal}")
else:
    print("No rate limits configured on this endpoint.")

In [0]:
current_user = w.current_user.me()
print(f"Email: {current_user.user_name}")

In [0]:
from databricks.sdk.service.serving import (
    AiGatewayRateLimit,
    AiGatewayRateLimitKey,
    AiGatewayRateLimitRenewalPeriod,
)

ENDPOINT_NAME = "databricks-claude-sonnet-4-6"

# Token-based rate limit: 100k tokens per minute for lucas.bruand only
response = w.serving_endpoints.put_ai_gateway(
    name=ENDPOINT_NAME,
    rate_limits=[
        AiGatewayRateLimit(
            key=AiGatewayRateLimitKey.USER,
            renewal_period=AiGatewayRateLimitRenewalPeriod.MINUTE,
            tokens=10000,
            principal=current_user.user_name,
        )
    ],
)

print(f"AI Gateway updated for endpoint: {ENDPOINT_NAME}")
print(f"Rate limits configured:")
for rl in response.rate_limits:
    print(f"  - Key: {rl.key.value}, Principal: {rl.principal}, Renewal: {rl.renewal_period.value}, Tokens: {rl.tokens}")

In [0]:
import requests

ctx = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
host = ctx.apiUrl().get()
token = ctx.apiToken().get()

url = f"{host}/serving-endpoints/{ENDPOINT_NAME}/invocations"
headers = {"Authorization": f"Bearer {token}", "Content-Type": "application/json"}
payload = {
    "messages": [{"role": "user", "content": "Say hello in one sentence."}],
    "max_tokens": 50,
}

resp = requests.post(url, headers=headers, json=payload, timeout=60)

print(f"Status: {resp.status_code}")
if resp.status_code == 200:
    body = resp.json()
    print(f"Response: {body['choices'][0]['message']['content']}")
    print(f"Usage: {body.get('usage')}")
else:
    print(f"Error: {resp.text[:500]}")

In [0]:
import time
import random
import string
import requests
from concurrent.futures import ThreadPoolExecutor
from databricks.sdk.service.serving import (
    AiGatewayRateLimit,
    AiGatewayRateLimitKey,
    AiGatewayRateLimitRenewalPeriod,
    AiGatewayUsageTrackingConfig,
)

# --- Config ---
TPM_VALUES = [1_000, 2_000, 5_000, 7_000, 10_000]
NUM_WORKERS = 10
RUN_TIME_SECONDS = 90  # per TPM value
COOLDOWN_SECONDS = 10 * 60  # wait between runs for the sliding window to reset

ctx = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
HOST = ctx.apiUrl().get()
TOKEN = ctx.apiToken().get()
URL = f"{HOST}/serving-endpoints/{ENDPOINT_NAME}/invocations"
HEADERS = {"Authorization": f"Bearer {TOKEN}", "Content-Type": "application/json"}

PROMPTS = [
    "Write a detailed 500-word essay covering the history of artificial intelligence, from its origins in the 1950s through modern deep learning breakthroughs.",
    "Explain the principles of quantum computing and how it differs from classical computing. Cover qubits, superposition, and entanglement in detail.",
    "Describe the evolution of the internet from ARPANET to the modern web. Include key milestones, protocols, and societal impacts.",
    "Discuss the causes and consequences of climate change, including the role of greenhouse gases, deforestation, and potential mitigation strategies.",
    "Write a comprehensive overview of the human immune system, covering innate and adaptive immunity, T-cells, B-cells, and vaccination mechanisms.",
    "Explain how blockchain technology works and its applications beyond cryptocurrency, including supply chain, healthcare, and voting systems.",
    "Describe the history of space exploration from Sputnik to the James Webb Space Telescope, highlighting major achievements and future goals.",
    "Discuss the impact of social media on modern society, including mental health effects, political polarization, and the spread of misinformation.",
]

def make_payload():
    nonce = ''.join(random.choices(string.ascii_lowercase + string.digits, k=12))
    prompt = random.choice(PROMPTS) + f" [ref:{nonce}]"
    return {"messages": [{"role": "user", "content": prompt}], "max_tokens": 1024}


def run_load_test(tpm_limit):
    """Set rate limit to tpm_limit and run load test. Returns (results_list, t0)."""
    # Set the rate limit
    w.serving_endpoints.put_ai_gateway(
        name=ENDPOINT_NAME,
        rate_limits=[
            AiGatewayRateLimit(
                key=AiGatewayRateLimitKey.USER,
                renewal_period=AiGatewayRateLimitRenewalPeriod.MINUTE,
                tokens=tpm_limit,
                principal=current_user.user_name,
            )
        ],
    )
    print(f"  Rate limit set to {tpm_limit:,} TPM")
    time.sleep(20)  # brief pause for config propagation

    test_results = []
    stop = False

    def send_request():
        while not stop:
            start = time.time()
            try:
                resp = requests.post(URL, headers=HEADERS, json=make_payload(), timeout=120)
                elapsed_ms = (time.time() - start) * 1000
                tokens = None
                if resp.status_code == 200:
                    tokens = resp.json().get("usage", {}).get("total_tokens")
                test_results.append((start, resp.status_code, elapsed_ms, tokens))
            except Exception:
                elapsed_ms = (time.time() - start) * 1000
                test_results.append((start, 0, elapsed_ms, None))

    run_t0 = time.time()
    with ThreadPoolExecutor(max_workers=NUM_WORKERS) as pool:
        futures = [pool.submit(send_request) for _ in range(NUM_WORKERS)]
        while time.time() - run_t0 < RUN_TIME_SECONDS:
            ok = sum(1 for r in test_results if r[1] == 200)
            limited = sum(1 for r in test_results if r[1] == 429)
            toks = sum(r[3] or 0 for r in test_results if r[1] == 200)
            print(f"    [{int(time.time()-run_t0):>3d}s] ok={ok}  429={limited}  tokens={toks}", end="\r")
            time.sleep(5)
        stop = True

    ok = sum(1 for r in test_results if r[1] == 200)
    limited = sum(1 for r in test_results if r[1] == 429)
    toks = sum(r[3] or 0 for r in test_results if r[1] == 200)
    print(f"    Done: {len(test_results)} requests, {ok} ok, {limited} 429s, {toks:,} tokens")
    return test_results, run_t0


# --- Run all TPM values ---
all_runs = {}  # {tpm: (results, t0)}

for i, tpm in enumerate(TPM_VALUES):
    print(f"\n{'='*60}")
    print(f"[{i+1}/{len(TPM_VALUES)}] Testing TPM = {tpm:,}")
    print(f"{'='*60}")
    run_results, run_t0 = run_load_test(tpm)
    all_runs[tpm] = (run_results, run_t0)

    # Cooldown between runs (except after the last one)
    if i < len(TPM_VALUES) - 1:
        print(f"  Cooling down {COOLDOWN_SECONDS}s for sliding window reset...")
        time.sleep(COOLDOWN_SECONDS)

# Remove rate limits after all tests
w.serving_endpoints.put_ai_gateway(
    name=ENDPOINT_NAME,
    rate_limits=[],
    usage_tracking_config=AiGatewayUsageTrackingConfig(enabled=True),
)
print(f"\nAll tests complete. Rate limits removed.")
print(f"Results stored in `all_runs` dict with keys: {list(all_runs.keys())}")

In [0]:
import csv
import os
from datetime import datetime

# Build rows from all_runs
rows = []
for tpm, (run_results, run_t0) in sorted(all_runs.items()):
    for ts, status, elapsed_ms, tokens in run_results:
        rows.append({
            "tpm_limit": tpm,
            "timestamp": ts,
            "relative_time_s": round(ts - run_t0, 3),
            "completion_time_s": round(ts + elapsed_ms / 1000 - run_t0, 3),
            "status_code": status,
            "elapsed_ms": round(elapsed_ms, 1),
            "total_tokens": tokens,
        })

# Save next to the notebook
notebook_dir = "/Workspace" + os.path.dirname(
    dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
)
timestamp_str = datetime.now().strftime("%Y%m%d_%H%M%S")
csv_path = f"{notebook_dir}/load_test_results_{timestamp_str}.csv"

with open(csv_path, "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=rows[0].keys())
    writer.writeheader()
    writer.writerows(rows)

print(f"Saved {len(rows):,} rows to:")
print(f"  {csv_path}")

In [0]:
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import numpy as np
from matplotlib.lines import Line2D
from matplotlib.patches import Patch

WINDOW_SEC = 60
COLORS = ["#E91E63", "#FF9800", "#4CAF50", "#2196F3", "#9C27B0"]

fig, axes = plt.subplots(len(all_runs), 1, figsize=(14, 4 * len(all_runs)), sharex=False)
if len(all_runs) == 1:
    axes = [axes]

for idx, (tpm, (run_results, run_t0)) in enumerate(sorted(all_runs.items())):
    ax = axes[idx]
    color = COLORS[idx % len(COLORS)]

    # Split by status, place tokens at completion time
    sorted_res = sorted(run_results, key=lambda r: r[0])
    success = [
        (r[0] + r[2] / 1000 - run_t0, r[3])
        for r in sorted_res if r[1] == 200 and r[3]
    ]
    limited = [r[0] - run_t0 for r in sorted_res if r[1] == 429]

    if not success:
        ax.text(0.5, 0.5, f"{tpm:,} TPM — no successful requests",
                transform=ax.transAxes, ha="center", va="center", fontsize=14)
        ax.set_title(f"Rate Limit: {tpm:,} TPM", fontsize=12, fontweight="bold")
        continue

    s_times = np.array([s[0] for s in success])
    s_tokens = np.array([s[1] for s in success])
    l_times = np.array(limited) if limited else np.array([])

    max_t = max(s_times[-1], l_times[-1]) if len(l_times) else s_times[-1]
    eval_times = np.arange(0, max_t + 1, 0.25)
    window_tokens = np.zeros_like(eval_times)
    for i, t in enumerate(eval_times):
        mask = (s_times >= t - WINDOW_SEC) & (s_times <= t)
        window_tokens[i] = s_tokens[mask].sum()

    # Sliding window
    ax.plot(eval_times, window_tokens, color=color, linewidth=2, zorder=3)
    ax.fill_between(eval_times, window_tokens, alpha=0.1, color=color)

    # Rate limit threshold
    ax.axhline(y=tpm, color="#F44336", linestyle="--", linewidth=1.5, zorder=4)

    # 429 markers
    if len(l_times):
        bucket_size = 5
        edges = np.arange(0, int(np.ceil(max_t / bucket_size)) * bucket_size + bucket_size, bucket_size)
        counts, _ = np.histogram(l_times, bins=edges)
        ax2 = ax.twinx()
        centers = (edges[:-1] + edges[1:]) / 2
        ax2.bar(centers, counts, width=bucket_size * 0.85, alpha=0.2, color="#FF9800", zorder=1)
        ax2.set_ylabel("429s / 5s", fontsize=9, color="#FF9800")
        ax2.tick_params(axis="y", labelcolor="#FF9800", labelsize=8)

    # Per-request scatter
    ax.scatter(s_times, s_tokens, color=color, s=15, alpha=0.6, zorder=5, edgecolors="white", linewidths=0.3)

    ok_count = sum(1 for r in sorted_res if r[1] == 200)
    lim_count = sum(1 for r in sorted_res if r[1] == 429)
    ax.set_title(f"Rate Limit: {tpm:,} TPM  —  {ok_count} ok · {lim_count:,} 429s",
                 fontsize=12, fontweight="bold")
    ax.set_ylabel("Tokens in 60s window", fontsize=9)
    ax.yaxis.set_major_formatter(ticker.FuncFormatter(lambda x, _: f"{x:,.0f}"))
    ax.grid(True, alpha=0.2)

axes[-1].set_xlabel("Time (seconds since start)", fontsize=12)

# Shared legend
legend_elements = [
    Line2D([0], [0], color="gray", linewidth=2, label="Inferred sliding window (60s) based on the HTTP transaction end"),
    Line2D([0], [0], color="#F44336", linestyle="--", linewidth=1.5, label="Rate limit threshold"),
    Line2D([0], [0], marker="o", color="w", markerfacecolor="gray", markersize=6, label="Per-request tokens"),
    Patch(facecolor="#FF9800", alpha=0.2, label="HTTP 429 Too Many Requests (rate-limited, 5s buckets)"),
]
fig.legend(handles=legend_elements, loc="upper center", ncol=2, fontsize=9,
           bbox_to_anchor=(0.5, 1.03), framealpha=0.9)

fig.suptitle("Sliding Window Token Consumption vs Rate Limit\nComparative Analysis Across TPM Thresholds",
             fontsize=14, fontweight="bold", y=1.08)
plt.tight_layout()
plt.show()